# 🏃 Physical Intensity & Tracking Analysis
## Notebook d'exploration — SkillCorner Open Data

**Objectif** : Explorer et comprendre les données de tracking SkillCorner

**Source** : [SkillCorner Open Data](https://github.com/SkillCorner/opendata) — 10 matchs, saison 2019/2020

**Plan du notebook** :
1. Charger les données d'un match
2. Explorer la structure des fichiers JSON
3. Calculer les métriques physiques
4. Visualiser les résultats
5. Analyse comparative entre joueurs

## 1. Installation et imports

In [ ]:
# Installation des librairies (si pas déjà fait)
# !pip install mplsoccer plotly pandas requests

import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from mplsoccer import Pitch
import warnings
warnings.filterwarnings('ignore')

# Ajouter le chemin parent pour importer src/
import sys
sys.path.append('..')
from src.data_loader import (
    MATCHES_INFO, load_match_metadata, build_player_speeds_df,
    compute_player_metrics, compute_speed_timeline, SPEED_ZONES
)

print('✅ Imports OK')

## 2. Liste des matchs disponibles

In [ ]:
# Afficher tous les matchs disponibles
print('📋 Matchs SkillCorner Open Data\n')
for match_id, info in MATCHES_INFO.items():
    print(f"  ID {match_id} : {info['home']} vs {info['away']} — {info['competition']} ({info['date']})")

## 3. Charger un match

In [ ]:
# On choisit Man City vs Liverpool (match_id = 2417)
# C'est l'un des matchs les plus emblématiques de la saison 2019/2020
MATCH_ID = 2417

print(f'⏳ Chargement du match {MATCH_ID}...')
print('   (peut prendre 1-2 minutes, les fichiers de tracking sont volumineux)')

# Charger les métadonnées
meta = load_match_metadata(MATCH_ID)
print(f'✅ Métadonnées chargées : {len(meta["players"])} joueurs')

# Charger et transformer les données de tracking
df_speeds = build_player_speeds_df(MATCH_ID)
print(f'✅ Tracking chargé : {len(df_speeds):,} frames × joueurs')

## 4. Explorer la structure des données

In [ ]:
# Structure du DataFrame de vitesses
print('📊 Colonnes disponibles :')
print(df_speeds.dtypes)
print()
print('📊 Premières lignes :')
df_speeds.head(10)

In [ ]:
# Statistiques de base
print('📊 Statistiques de vitesse :')
print(df_speeds['speed_kmh'].describe())
print(f'\n🏃 Joueurs uniques : {df_speeds["player_id"].nunique()}')
print(f'📸 Frames totales : {df_speeds["frame"].nunique():,}')
print(f'⏱️  Durée estimée : {df_speeds["frame"].nunique() / (10*60):.1f} minutes')

## 5. Calculer les métriques physiques

In [ ]:
# Calculer toutes les métriques par joueur
metrics = compute_player_metrics(df_speeds)

print('✅ Métriques calculées pour', len(metrics), 'joueurs')
metrics[['player_name', 'team_name', 'total_distance_km', 'max_speed_kmh', 'sprint_count', 'hir_distance_m']].head(10)

## 6. Visualisation — Top coureurs

In [ ]:
# Graphique barres horizontales
top15 = metrics.head(15).copy()
top15['label'] = top15['player_name'] + ' (' + top15['team_name'] + ')'

fig, ax = plt.subplots(figsize=(10, 8), facecolor='#0A0A0A')
ax.set_facecolor('#111111')

colors = ['#00FF87' if t == top15['team_name'].iloc[0] else '#FF6B35' 
          for t in top15['team_name']]

bars = ax.barh(top15['label'], top15['total_distance_km'], color=colors, alpha=0.85)

for bar, val in zip(bars, top15['total_distance_km']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f} km', va='center', color='white', fontsize=9)

ax.set_xlabel('Distance (km)', color='white')
ax.set_title('Distance totale — Top 15 joueurs', color='white', fontsize=14, pad=10)
ax.tick_params(colors='white')
ax.spines['bottom'].set_color('#333')
ax.spines['left'].set_color('#333')
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

## 7. Profil de vitesse d'un joueur

In [ ]:
# Sélectionner le joueur avec le plus de distance
top_player_id = metrics.iloc[0]['player_id']
top_player_name = metrics.iloc[0]['player_name']

# Extraire sa timeline de vitesse
timeline = compute_speed_timeline(df_speeds, top_player_id)

# Lissage
timeline['speed_smooth'] = timeline['speed_kmh'].rolling(10, center=True).mean()

# Graphique
fig = go.Figure()

for period, color in [(1, '#00FF87'), (2, '#FF6B35')]:
    p_data = timeline[timeline['period'] == period]
    fig.add_trace(go.Scatter(
        x=p_data['minutes'], y=p_data['speed_smooth'],
        name=f'MT {period}', line={'color': color, 'width': 1.5},
        fill='tozeroy'
    ))

fig.add_hline(y=23, line_dash='dot', line_color='red',
              annotation_text='Sprint (23 km/h)')

fig.update_layout(
    title=f'Profil de vitesse — {top_player_name}',
    xaxis_title='Temps (min)', yaxis_title='Vitesse (km/h)',
    paper_bgcolor='#0A0A0A', plot_bgcolor='#111111',
    font={'color': 'white'}, height=400
)
fig.show()

## 8. Heatmap de positions

In [ ]:
# Heatmap du joueur sélectionné
player_df = df_speeds[
    (df_speeds['player_id'] == top_player_id) &
    df_speeds['x'].notna() & df_speeds['y'].notna()
]

pitch = Pitch(pitch_type='skillcorner', pitch_color='#0A1628',
              line_color='#3A7BD5', linewidth=1.5)

fig, ax = plt.subplots(figsize=(12, 8), facecolor='#0A0A0A')
pitch.draw(ax=ax)

pitch.kdeplot(player_df['x'], player_df['y'], ax=ax,
              cmap='hot', fill=True, alpha=0.7, levels=50, bw_adjust=0.5)

ax.set_title(f'Heatmap — {top_player_name}',
             color='white', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 9. Comparaison par zones d'intensité

In [ ]:
# Zones pour les 10 premiers joueurs
zone_cols = [f'dist_{z}' for z in SPEED_ZONES.keys() if f'dist_{z}' in metrics.columns]
top10 = metrics.head(10).copy()
top10['label'] = top10['player_name'].str.split().str[-1]

zone_colors = ['#4CAF50', '#8BC34A', '#FFC107', '#FF5722', '#F44336']
zone_labels = list(SPEED_ZONES.keys())

fig, ax = plt.subplots(figsize=(14, 7), facecolor='#0A0A0A')
ax.set_facecolor('#111111')

bottoms = np.zeros(len(top10))
for zone, color in zip(zone_cols, zone_colors):
    if zone in top10.columns:
        ax.bar(top10['label'], top10[zone], bottom=bottoms,
               color=color, alpha=0.85, label=zone.replace('dist_', ''))
        bottoms += top10[zone].values

ax.set_xlabel('Joueur', color='white')
ax.set_ylabel('Distance (m)', color='white')
ax.set_title('Zones d\'intensité — Top 10 joueurs', color='white', fontsize=14)
ax.tick_params(colors='white', rotation=30)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left',
          facecolor='#222', labelcolor='white')

for spine in ax.spines.values():
    spine.set_color('#333')

plt.tight_layout()
plt.show()

print('\n📊 Métriques complètes disponibles dans metrics.columns :')
print([c for c in metrics.columns])